# Les tests d'hypothèse avec Python

### Dans ce live-coding, nous allons examiner le déroulement d'un test d'hypothèse et tenter de comprendre **l'intuition graphique** derrière la `p-valeur`.

#### Mais d'abord, qu'est-ce qu'un test d'hypothèse ?

**Un test d'hypothèse est une méthode statistique utilisée pour évaluer une affirmation ou une hypothèse sur une population à partir d'un échantillon de données.**

> _L'objectif est de **déterminer si les données fournissent suffisamment de preuves** pour rejeter une hypothèse initiale, **appelée hypothèse nulle,** en faveur d'une hypothèse alternative._

En Python, des bibliothèques comme SciPy et NumPy sont couramment utilisées pour effectuer des tests d'hypothèse. Ces tests permettent de **comparer des moyennes, des proportions ou des distributions, et ainsi de prendre des décisions objectives sur les données.**


### Mais avant d'aller plus loin, comme toujours nous importons les **packages** nécéssaires.

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns

from scipy import stats

In [ ]:
# Initialisation d'un générateur de nombres aléatoires avec une graine spécifique (10) à l'aide de la méthode '.default_rng()'.

rng = np.random.default_rng(10)

**À quoi ça sert ?**

Ici, `rng` est un objet de type **générateur** qui permet de **produire des nombres aléatoires**.

La graine (ou seed en anglais) `(10)` permet de garantir que les **résultats aléatoires** générés sont **reproductibles** à chaque exécution du code.




## La génération de lois normales

### Les **lois normales** se retrouvent dans de nombreux phénomènes du quotidien.

> _Grâce au **Théorème Central Limite**, à partir d'une **distribution de départ inconnue**, nous pouvons nous ramener à une **loi normale** dont nous connaissons les propriétés. Ensuite, nous sommes capables de calculer la `p-valeur`, qui a une **interprétation graphique intuitive.**_

#### Il est important de rappeler que l'on **respecte trois règles** :

- Les points de données doivent être **générés indépendamment**.

- Les points de données doivent être **tirés de la même distribution** : cela signifie que la situation globale que l'on observe, comme un retard de train ou le comportement d'un utilisateur, ne doit pas changer.

> _Par exemple, si le site internet reste stable et que la situation économique ne change pas trop, nous pouvons **raisonnablement supposer** que la loi qui régit les **comportements des utilisateurs** reste stable._

- Les **points de données** doivent être **suffisamment nombreux**. On peut considérer que **100 points** ou plus sont suffisants, mais plus nous ajoutons de points, plus la courbe finale ressemblera à une **loi normale.**

**Générons deux distributions correspondant, dans notre exemple, à deux paniers moyens différents. Supposons que nous estimions la moyenne 1 comme étant celle du site 1, et la moyenne 2 comme étant celle du nouveau site.**

In [ ]:
panier_1 = rng.normal(loc=20, scale=2, size=1000) # Stockons 1000 valeurs autour de 20 avec un écart-type de 2.
panier_2 = rng.normal(loc=25, scale=2, size=1000) # Stockons 1000 valeurs autour de 25 avec un écart-type de 2.


**Visualisons les dix premières valeurs de nos deux arrays.**

In [ ]:
panier_1[:10]

In [ ]:
panier_2[:10]

On peut voir ici que les valeurs **tournent autour** de la moyenne que l'on a fixée (20 pour le premier panier, 25 pour le deuxième), et qu'elles **évoluent en moyenne** autour de + ou - 2 autour de 20 et 25 respectivement.

In [ ]:
test = stats.ttest_ind(panier_1, panier_2, random_state=rng) # La méthode '.ttest_ind()' nous permet de renseigner nos deux arrays, il s'agit de notre test d'hypothèse.
print(test)

In [ ]:
test.pvalue # Nous pouvons ressortir uniquement la p-value avec la méthode '.pvalue'

#### **Le test réalisé ici s'effectue très rapidement.**

On observe une **p-valeur nulle**, ce qui correspond à notre **intuition** selon laquelle **le nouveau site a fait augmenter le panier moyen sans que cela soit dû à un hasard.**

> _En parlant de hasard, il est possible que **deux ou trois acheteurs** soient redirigés vers le site 2 par chance et qu'ils achètent des **quantités astronomiques de produits.** C'est peu probable, **mais c'est envisageable !**_

Dans cet exemple, la **p-valeur** indique la **probabilité que la différence de moyenne soit attribuable à ces trois acheteurs**, qui n'ont pas dépensé plus à cause du nouveau site, mais simplement **parce qu'ils avaient envie d'acheter pour des raisons qui n'ont rien à voir avec le changement.**

#### **Ici, une p-valeur nulle indique avec confiance qu'il est quasi impossible que cela soit le cas, et que c'est donc bien le nouveau site qui a provoqué ce changement.**

### Explication graphique de la p-valeur

La fonction suivante va nous **permettre** de représenter **graphiquement deux lois normales** et de réaliser un test sur la différence entre ces lois, **c'est ce qui se passe en pratique dans nos tests d'hypothèse.**

La **p-valeur** représente la **probabilité que les valeurs observées pour le panier 2 soient dues à un hasard.** Ainsi, plus les **courbes sont éloignées**, moins il est probable que les **valeurs observées** sur le panier 2 ne soient que des **valeurs extrêmes** du panier 1.

#### Voyons pourquoi.

In [ ]:
# Cette fonction génère deux distributions normales, compare leurs histogrammes avec une estimation de densité de noyau, et affiche la p-valeur d'un test t de Student entre elles, rentrons dans le détail :

def generate_normal_laws(mu_1, mu_2, sample_size):
    # Génère une distribution normale autour de mu_1 avec un écart-type de 2.
    distribution_1 = rng.normal(loc=mu_1, scale=2, size=sample_size)

    # Génère une distribution normale autour de mu_2 avec un écart-type de 2.
    distribution_2 = rng.normal(loc=mu_2, scale=2, size=sample_size)

    # Définit la taille de la figure pour les graphiques.
    sns.set(rc={'figure.figsize':(6,6)})

    # Crée un histogramme de distribution_1 avec une estimation de densité de noyau.
    ax = sns.histplot(x=distribution_1, kde=True, stat='density', label='samples')

    # Définit le titre du graphique comparant les deux lois normales.
    ax.set_title(f'Comparaison des lois normales N({mu_1}, 2) et N({mu_2}, 2), n={sample_size} tirages')

    # Définit l'étiquette de l'axe des x.
    ax.set_xlabel("Panier moyen")

    # Définit l'étiquette de l'axe des y.
    ax.set_ylabel("Proportion dans la population")

    # Ajoute un histogramme de distribution_2 sur le même graphique.
    sns.histplot(x=distribution_2, kde=True, stat='density', label='samples', ax=ax)

    # Affiche la p-valeur d'un test t de Student comparant les deux distributions.
    print(f"P-valeur du test : {stats.ttest_ind(distribution_1, distribution_2, random_state=rng).pvalue}")

### **1er cas :** _moyennes très éloignées._

**Ici, imaginons que le panier 1 vaut 20€ et le second 25€. Prenons 1000 achats de clients sur chaque site et voyons ce que cela donne.**

In [ ]:
# Initialisation de valeurs pour pouvoir les rentrer dans la fonction :

mu_1 = 20 # Panier moyen 1.
mu_2 = 25 # Panier moyen 2.
sample_size = 1000 # Taille des échantillons.

generate_normal_laws(mu_1, mu_2, sample_size) # Application de la fonction.

#### **La p-valeur est nulle ! Pourquoi ?**

> Si je vous demandais si **les deux courbes étaient superposées**, vous me répondriez **"Non !"** immédiatement.

C'est ce qui ressort du **test d'hypothèse**, les valeurs entre **27.5 et 30.0** ne peuvent pas provenir de la **distribution bleue**, car elles sont **trop à droite** sur la courbe.

**Conséquence, elles doivent provenir d'une nouvelle distribution, la courbe orange !**

#### **Maintenant, que ce passe t-il si nous rapprochons les moyennes ?**

### **2ème cas :** _moyennes moins éloignées._

In [ ]:
# Initialisation de valeurs pour pouvoir les rentrer dans la fonction :

mu_1 = 20
mu_2 = 20.5
sample_size = 1000

generate_normal_laws(mu_1, mu_2, sample_size)

Ici, l'oeil humain n'est plus **assez bon** pour distinguer si les **valeurs de la courbe orange** peuvent raisonnablement **provenir de la courbe bleue**.

#### On peut même instinctivement se dire qu'elles proviennent de la courbe bleue et qu'il n'y a pas d'autre distribution. **Erreur !**

> Ici, **la p-valeur troune autour de 10^-9 :** à un niveau de confiance 95%, on rejette immédiatement l'hypothèse nulle : **les valeurs viennent bien de deux distributions différentes avec un haut niveau de confiance !**

### **3ème cas :** _moyennes rapprochées._

In [ ]:
# Initialisation de valeurs pour pouvoir les rentrer dans la fonction :

mu_1 = 20
mu_2 = 20.1
sample_size = 1000

generate_normal_laws(mu_1, mu_2, sample_size)

**Ici, la p-valeur atteint environ 86%.**

#### Dans la plupart des cas, on ne rejettera pas l'hypothèse.

> Il est ici impossible de savoir si le **changement de site** explique l'écart de **0.1** entre les paniers moyens, ou si cela est **dû encore une fois au hasard.**

#### Cependant, que se passe-t-il si **j'augmente massivement** mon nombre de points de données ?

In [ ]:
# Initialisation de valeurs pour pouvoir les rentrer dans la fonction :

mu_1 = 20
mu_2 = 20.1
sample_size = 100_000

generate_normal_laws(mu_1, mu_2, sample_size)

En multipliant le nombre de lignes de données par 1000, nous sommes passés de 1000 à 100 000 lignes, et la p-valeur **s'est abaissée à environ `10e-26`, soit quasiment zéro !**

> C'est l'illustration du caractère impitoyable du **Théorème Central Limite** : en **augmentant** le nombre de points de données, nous remarquons que les **deux courbes sont désormais des lois normales quasi parfaites**, encore plus **lissées** que précédemment.

**Le test n'a donc plus aucun doute sur le fait que la différence de moyenne est due au changement de site et non au hasard.**

En clair, si un test statistique ne fonctionne pas et que la p-valeur est trop élevée, **il faut chercher à obtenir plus de données ou des données de meilleure qualité.** Même dans une situation où la différence des moyennes est faible, le test sera plus robuste avec davantage de données.

Cela s'interprète par le fait que même des **clients ayant des achats élevés ou compulsifs** ne suffisent plus à influencer le résultat.

#### De ce fait, les erreurs se compensent et nous obtenons un **résultat clair sur l'efficacité du site.**